In [1]:
from investment_agents.data.clients.finmind import FinMindClient

from investment_agents.data.repositories.price import PriceRepository
from investment_agents.data.repositories.dividend import DividendRepository
from investment_agents.data.repositories.institutional import InstitutionalRepository
from investment_agents.data.repositories.market_regime import MarketRegimeRepository

from investment_agents.features.price_adjustment import PriceAdjustmentService
from investment_agents.features.technical import TechnicalFeatureService
from investment_agents.features.chip import ChipFeatureService
from investment_agents.features.regime import RegimeFeatureService

from investment_agents.snapshots.technical import TechnicalSnapshotService
from investment_agents.snapshots.chip import ChipSnapshotService
from investment_agents.snapshots.regime import MarketRegimeSnapshotService

from investment_agents.agents.technical import TechnicalAgent
from investment_agents.agents.chip import ChipAgent
from investment_agents.agents.regime import MarketRegimeAgent
from investment_agents.agents.portfolio_manager import PortfolioManagerAgent

from investment_agents.ranking.daily import DailyRankingService

2026-09-21 01:28:40.819 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success
2026-09-21 01:28:40.866 | INFO     | FinMind.data.finmind_api:login_by_token:85 - Login success


In [2]:
client = FinMindClient()

price_repository = PriceRepository(client)
dividend_repository = DividendRepository(client)
institutional_repository = InstitutionalRepository(client)
regime_repository = MarketRegimeRepository(client)

In [3]:
price_adjustment_service = PriceAdjustmentService()

technical_feature_service = TechnicalFeatureService()

chip_feature_service = ChipFeatureService()

regime_feature_service = RegimeFeatureService()

In [4]:
technical_snapshot_service = TechnicalSnapshotService(
    price_repo=price_repository,
    dividend_repo=dividend_repository,
    adjustment_service=price_adjustment_service,
    technical_service=technical_feature_service,
)

chip_snapshot_service = ChipSnapshotService(
    institutional_repo=institutional_repository,
    price_repo=price_repository,
    chip_service=chip_feature_service,
)

regime_snapshot_service = MarketRegimeSnapshotService(
    repository=regime_repository,
    feature_service=regime_feature_service,
)

In [5]:
technical_agent = TechnicalAgent()
chip_agent = ChipAgent()
regime_agent = MarketRegimeAgent()
pm_agent = PortfolioManagerAgent()

ranking_service = DailyRankingService(
    technical_snapshot_service=technical_snapshot_service,
    chip_snapshot_service=chip_snapshot_service,
    regime_snapshot_service=regime_snapshot_service,

    technical_agent=technical_agent,
    chip_agent=chip_agent,
    regime_agent=regime_agent,
    pm_agent=pm_agent,
)

In [6]:
test_universe = [
    "2330",
    "2454",
    "2317",
]

as_of_date = "2026-09-18"

In [7]:
ranking_df = ranking_service.run(
    tickers=test_universe,
    as_of_date=as_of_date,
)

ranking_df

2026-09-21 01:28:42.273 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalReturnIndex, data_id: TAIEX
2026-09-21 01:28:42.449 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockTotalInstitutionalInvestors, data_id: 
2026-09-21 01:28:46.571 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-21 01:28:46.655 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockDividendResult, data_id: 2330
2026-09-21 01:28:46.740 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2330
2026-09-21 01:28:46.818 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2330
2026-09-21 01:28:53.751 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2454
2026-09-21 01:28:53.861 | INFO     | FinMind.data.finmi

,as_of_date,ticker,technical_score,chip_score,regime,regime_score,regime_confidence,final_score,conviction,rank
0,2026-09-18,2330,75,50,risk_on,75,80,75,70,1
1,2026-09-18,2454,85,50,risk_on,75,80,75,70,2
2,2026-09-18,2317,60,0,risk_on,75,80,40,50,3


In [8]:
chip_2317 = chip_snapshot_service.get_snapshot(
    ticker="2317",
    as_of_date="2026-09-18",
)

chip_2317

2026-09-21 01:31:42.780 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-09-21 01:31:42.957 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2317


{'foreign_flow_ratio_1d': -14.949026872393723,
 'foreign_flow_ratio_5d': -1.3012731278793255,
 'foreign_flow_ratio_20d': -0.32449807306897616,
 'foreign_buy_days_20d': 7.0,
 'foreign_streak': -1.0,
 'trust_flow_ratio_1d': -1.2145784410490903,
 'trust_flow_ratio_5d': 1.1770034172373036,
 'trust_flow_ratio_20d': 0.7541243186715478,
 'trust_buy_days_20d': 13.0,
 'trust_streak': -1.0,
 'dealer_flow_ratio_1d': 1.9561139016287639,
 'dealer_flow_ratio_5d': -1.173553940418045,
 'dealer_flow_ratio_20d': -0.3599471607566616,
 'dealer_buy_days_20d': 11.0,
 'dealer_streak': 2.0}

In [9]:
chip_data = chip_snapshot_service.get_snapshot(
    ticker="2317",
    as_of_date="2026-09-18",
)

chip_report = chip_agent.analyze(chip_data)

print("score:", chip_report.score)
print("reason:", chip_report.reason)

2026-09-21 01:32:35.761 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockPrice, data_id: 2317
2026-09-21 01:32:35.842 | INFO     | FinMind.data.finmind_api:get_data:166 - download Dataset.TaiwanStockInstitutionalInvestorsBuySell, data_id: 2317


score: 0
reason: Foreign Investors show strong short-term selling pressure with a 1-day flow ratio of -14.95% and a negative 5-day flow ratio, indicating persistent selling. Investment Trusts have a mixed signal with a positive 20-day flow ratio but recent selling, as shown by the negative 1-day flow ratio and current streak. Dealers show a short-term buying signal with a positive 1-day flow ratio and a positive streak, but their medium-term flow is negative. The divergence between strong Foreign Investor selling and mixed signals from Investment Trusts and Dealers suggests unattractive conditions.
